# LeetCode #1340: Jump Game V

https://leetcode.com/problems/jump-game-v/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ (without memoization) | $O(n)$ |
| **Optimal: Sort + Memoized DFS ★** | $O(n \cdot d)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
From each index, try all valid jumps recursively without caching results. The same sub-problems are recomputed many times, wasting work exponentially in the worst case.

### Optimal: Sort + Memoized DFS ★
Sort indices by height (shortest first) so that when we compute `dp[i]` all reachable positions (which must be strictly lower) are already computed. For each index `i`, scan left and right up to `d` steps, stopping if a bar at least as tall as `arr[i]` is encountered. `dp[i] = 1 + max(dp[j])` over all valid `j`.

**Why this is better than Brute Force:** Sorting guarantees each state is computed at most once. Total work is $O(n \log n + n \cdot d)$, dominated by the sort and the bounded scan.

**Constraints:**
* $1 \leq$ `arr.length` $\leq 1000$
* $1 \leq$ `arr[i]` $\leq 10^5$
* $1 \leq d \leq$ `arr.length`

## Solutions

### C#

In [ ]:
using System.Linq;

public class Solution {
    public int MaxJumps(int[] arr, int d) {
        int n = arr.Length;
        int[] dp = new int[n]; // dp[i] = max indices visitable starting from i

        // Process in order of increasing height so lower bars are already solved
        foreach (int i in Enumerable.Range(0, n).OrderBy(i => arr[i])) {
            dp[i] = 1; // Always visit the starting index itself
            // Scan right: stop at a bar >= arr[i] (can't jump over or onto it)
            for (int j = i + 1; j <= Math.Min(i + d, n - 1) && arr[j] < arr[i]; j++)
                dp[i] = Math.Max(dp[i], 1 + dp[j]);
            // Scan left: same stopping rule
            for (int j = i - 1; j >= Math.Max(i - d, 0) && arr[j] < arr[i]; j--)
                dp[i] = Math.Max(dp[i], 1 + dp[j]);
        }
        return dp.Max();
    }
}

### Python

In [ ]:
from typing import List

class Solution:
    def max_jumps(self, arr: List[int], d: int) -> int:
        n = len(arr)
        dp = [1] * n  # dp[i] = max indices visitable starting from i

        # Process in order of increasing height so lower bars are already solved
        for i in sorted(range(n), key=lambda x: arr[x]):
            # Scan right: stop at a bar >= arr[i]
            for j in range(i + 1, min(i + d + 1, n)):
                if arr[j] >= arr[i]:
                    break
                dp[i] = max(dp[i], 1 + dp[j])
            # Scan left: same stopping rule
            for j in range(i - 1, max(i - d - 1, -1), -1):
                if arr[j] >= arr[i]:
                    break
                dp[i] = max(dp[i], 1 + dp[j])

        return max(dp)

### Go

In [ ]:
import "sort"

func maxJumps(arr []int, d int) int {
    n := len(arr)
    dp := make([]int, n) // dp[i] = max indices visitable starting from i
    for i := range dp { dp[i] = 1 }

    // Process in order of increasing height so lower bars are already solved
    order := make([]int, n)
    for i := range order { order[i] = i }
    sort.Slice(order, func(a, b int) bool { return arr[order[a]] < arr[order[b]] })

    for _, i := range order {
        // Scan right: stop at a bar >= arr[i]
        for j := i + 1; j <= i+d && j < n; j++ {
            if arr[j] >= arr[i] { break }
            if 1+dp[j] > dp[i] { dp[i] = 1 + dp[j] }
        }
        // Scan left: same stopping rule
        for j := i - 1; j >= i-d && j >= 0; j-- {
            if arr[j] >= arr[i] { break }
            if 1+dp[j] > dp[i] { dp[i] = 1 + dp[j] }
        }
    }
    best := 0
    for _, v := range dp { if v > best { best = v } }
    return best
}

### Rust

In [ ]:
impl Solution {
    pub fn max_jumps(arr: Vec<i32>, d: i32) -> i32 {
        let n = arr.len();
        let d = d as usize;
        let mut dp = vec![1i32; n]; // dp[i] = max indices visitable starting from i

        // Process in order of increasing height so lower bars are already solved
        let mut order: Vec<usize> = (0..n).collect();
        order.sort_unstable_by_key(|&i| arr[i]);

        for i in order {
            // Scan right: stop at a bar >= arr[i]
            for j in (i + 1)..=(i + d).min(n - 1) {
                if arr[j] >= arr[i] { break; }
                dp[i] = dp[i].max(1 + dp[j]);
            }
            // Scan left: same stopping rule
            let lo = if i >= d { i - d } else { 0 };
            for j in (lo..i).rev() {
                if arr[j] >= arr[i] { break; }
                dp[i] = dp[i].max(1 + dp[j]);
            }
        }
        *dp.iter().max().unwrap()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [6,4,14,6,8,13,9,7,10,6,12], d = 2`
The tallest bar (14 at index 2) can reach only indices 1 and 3 (both lower). From index 2 the chain extends further. Maximum: **4**.

### 2. Slightly Complex
**Input:** `arr = [3,3,3,3,3], d = 3`
All heights are equal — no jump satisfies `arr[j] < arr[i]`. Every index is isolated and returns **1**.

### 3. Edge Case: Time Factor
**Input:** `arr` of 1000 strictly increasing values, `d = 1000`
The tallest index (last) scans the entire array and all sub-problems must be solved first. Total work is $O(n \cdot d) = O(n^2)$ — worst-case runtime.

### 4. Edge Case: Space Factor
**Input:** `n = 1000`
The `dp` array holds 1000 entries. Stack depth for the iterative implementation is $O(1)$ beyond the array allocation.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [100000, 1, 100000], d = 1`
Index 1 (height 1) can jump left to index 0 only if arr[0] < 1 — but arr[0]=100000 > 1, so no jump. Same right. dp = [1,1,1]. Answer: **1**.